# ElasticNet + ExtraTrees + RandomForest Stacking


In [2]:
%pip install -q numerapi pyarrow scikit-learn scipy joblib cloudpickle numerai-tools optuna

Note: you may need to restart the kernel to use updated packages.


In [4]:
# ============================================
# 0. 라이브러리 / 경로
# ============================================

import numpy as np
import pandas as pd
from pathlib import Path
from numerai_tools.scoring import numerai_corr

BASE_DIR = Path("/Users/seomichelle/26-1 ESAA:Python/datasets/Numerai")
CACHE_DIR = BASE_DIR / "cache_v52"
DIAGNOSTICS_DIR = BASE_DIR / "diagnostics_v52"

DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

RF_PATH = BASE_DIR / "rf_fullrefit_diagnostics.csv"
ET_PATH = BASE_DIR / "et_fullrefit_diagnostics.csv"
ELASTIC_PATH = BASE_DIR / "elasticnet_fullrefit_diagnostics.csv"

VALIDATION_INFO_PATH = CACHE_DIR / "validation_pca500_lag1_lag2.parquet"

DIAGNOSTICS_CSV = DIAGNOSTICS_DIR / "elasticnet_extratrees_randomforest_v52_diagnostics.csv"
PER_ERA_CSV = DIAGNOSTICS_DIR / "elasticnet_et_rf_per_era_corr.csv"
SUMMARY_CSV = DIAGNOSTICS_DIR / "elasticnet_et_rf_local_summary.csv"

print("RF:", RF_PATH)
print("ET:", ET_PATH)
print("ElasticNet:", ELASTIC_PATH)

RF: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/rf_fullrefit_diagnostics.csv
ET: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/et_fullrefit_diagnostics.csv
ElasticNet: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/elasticnet_fullrefit_diagnostics.csv


In [5]:
# ============================================
# 1. 예측 CSV 및 공식 validation 정보 불러오기
# ============================================

rf = pd.read_csv(RF_PATH)
et = pd.read_csv(ET_PATH)
elastic = pd.read_csv(ELASTIC_PATH)

rf["id"] = rf["id"].astype(str)
et["id"] = et["id"].astype(str)
elastic["id"] = elastic["id"].astype(str)

validation_info = pd.read_parquet(
    VALIDATION_INFO_PATH,
    columns=["era", "target"],
).reset_index()

validation_info = validation_info.rename(columns={validation_info.columns[0]: "id"})
validation_info["id"] = validation_info["id"].astype(str)
validation_info["row_order"] = np.arange(len(validation_info))

print("RF shape:", rf.shape)
print("ET shape:", et.shape)
print("ElasticNet shape:", elastic.shape)
print("Validation shape:", validation_info.shape)

print("RF = ET ID 순서:", rf["id"].equals(et["id"]))
print("RF = ElasticNet ID 순서:", rf["id"].equals(elastic["id"]))

display(rf.head())
display(et.head())
display(elastic.head())

RF shape: (4085748, 2)
ET shape: (4085748, 2)
ElasticNet shape: (4050241, 2)
Validation shape: (4050241, 4)
RF = ET ID 순서: True
RF = ElasticNet ID 순서: False


,id,prediction
0,n000101811a8a843,0.500177
1,n001e1318d5072ac,0.502300
2,n002a9c5ab785cbb,0.500931
3,n002ccf6d0e8c5ad,0.500500
4,n0041544c345c91d,0.501454


,id,prediction
0,n000101811a8a843,0.500239
1,n001e1318d5072ac,0.501182
2,n002a9c5ab785cbb,0.500023
3,n002ccf6d0e8c5ad,0.500754
4,n0041544c345c91d,0.500529


,id,prediction
0,n000101811a8a843,0.386445
1,n001e1318d5072ac,0.784514
2,n002a9c5ab785cbb,0.205114
3,n002ccf6d0e8c5ad,0.532547
4,n0041544c345c91d,0.467096


In [6]:
# ============================================
# 2. ID 기준 정렬 및 병합
# ============================================

rf = rf.rename(columns={"prediction": "rf_prediction"})
et = et.rename(columns={"prediction": "et_prediction"})
elastic = elastic.rename(columns={"prediction": "elasticnet_prediction"})

prediction_df = validation_info.merge(
    rf[["id", "rf_prediction"]],
    on="id",
    how="left",
    validate="one_to_one",
)

prediction_df = prediction_df.merge(
    et[["id", "et_prediction"]],
    on="id",
    how="left",
    validate="one_to_one",
)

prediction_df = prediction_df.merge(
    elastic[["id", "elasticnet_prediction"]],
    on="id",
    how="left",
    validate="one_to_one",
)

prediction_df = prediction_df.sort_values("row_order").reset_index(drop=True)

print("병합 결과:", prediction_df.shape)
print("RF 결측치:", prediction_df["rf_prediction"].isna().sum())
print("ET 결측치:", prediction_df["et_prediction"].isna().sum())
print("ElasticNet 결측치:", prediction_df["elasticnet_prediction"].isna().sum())

assert prediction_df["rf_prediction"].notna().all()
assert prediction_df["et_prediction"].notna().all()
assert prediction_df["elasticnet_prediction"].notna().all()

display(prediction_df.head())

병합 결과: (4050241, 7)
RF 결측치: 0
ET 결측치: 0
ElasticNet 결측치: 0


,id,era,target,row_order,rf_prediction,et_prediction,elasticnet_prediction
0,n000101811a8a843,575,0.5,0,0.500177,0.500239,0.386445
1,n001e1318d5072ac,575,0.5,1,0.502300,0.501182,0.784514
2,n002a9c5ab785cbb,575,0.5,2,0.500931,0.500023,0.205114
3,n002ccf6d0e8c5ad,575,0.0,3,0.500500,0.500754,0.532547
4,n0041544c345c91d,575,0.5,4,0.501454,0.500529,0.467096


In [7]:
# ============================================
# 3. ElasticNet + ExtraTrees + RandomForest rank average
# ============================================

prediction_df["rf_rank"] = prediction_df.groupby("era")["rf_prediction"].rank(
    method="average",
    pct=True,
).astype(np.float32)

prediction_df["et_rank"] = prediction_df.groupby("era")["et_prediction"].rank(
    method="average",
    pct=True,
).astype(np.float32)

prediction_df["elasticnet_rank"] = prediction_df.groupby("era")["elasticnet_prediction"].rank(
    method="average",
    pct=True,
).astype(np.float32)

prediction_df["prediction"] = prediction_df[
    ["rf_rank", "et_rank", "elasticnet_rank"]
].mean(axis=1).astype(np.float32)

diagnostics_df = prediction_df[["id", "prediction"]].copy()
diagnostics_df.to_csv(DIAGNOSTICS_CSV, index=False)

print("Diagnostics CSV:", DIAGNOSTICS_CSV)
print("shape:", diagnostics_df.shape)
print("prediction min:", diagnostics_df["prediction"].min())
print("prediction max:", diagnostics_df["prediction"].max())

display(diagnostics_df.head())

Diagnostics CSV: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/diagnostics_v52/elasticnet_extratrees_randomforest_v52_diagnostics.csv
shape: (4050241, 2)
prediction min: 0.0002216312
prediction max: 1.0


,id,prediction
0,n000101811a8a843,0.534931
1,n001e1318d5072ac,0.903732
2,n002a9c5ab785cbb,0.506378
3,n002ccf6d0e8c5ad,0.685801
4,n0041544c345c91d,0.709347


In [8]:
# ============================================
# 4. ID 순서 및 예측값 확인
# ============================================

print("최종 CSV = RF ID 순서:", diagnostics_df["id"].equals(rf["id"]))
print("최종 CSV = ET ID 순서:", diagnostics_df["id"].equals(et["id"]))
print("최종 CSV = ElasticNet ID 순서:", diagnostics_df["id"].equals(elastic["id"]))

comparison = prediction_df[
    [
        "id",
        "era",
        "rf_prediction",
        "et_prediction",
        "elasticnet_prediction",
        "rf_rank",
        "et_rank",
        "elasticnet_rank",
        "prediction",
    ]
].head(10)

display(comparison)

최종 CSV = RF ID 순서: False
최종 CSV = ET ID 순서: False
최종 CSV = ElasticNet ID 순서: True


,id,era,rf_prediction,et_prediction,elasticnet_prediction,rf_rank,et_rank,elasticnet_rank,prediction
0,n000101811a8a843,575,0.500177,0.500239,0.386445,0.563841,0.654506,0.386445,0.534931
1,n001e1318d5072ac,575,0.502300,0.501182,0.784514,0.971924,0.954757,0.784514,0.903732
2,n002a9c5ab785cbb,575,0.500931,0.500023,0.205114,0.774678,0.539342,0.205114,0.506378
3,n002ccf6d0e8c5ad,575,0.500500,0.500754,0.532547,0.660587,0.864270,0.532547,0.685801
4,n0041544c345c91d,575,0.501454,0.500529,0.467096,0.877146,0.783798,0.467096,0.709347
5,n0051ab821295c29,575,0.498854,0.498906,0.205472,0.174893,0.072425,0.205472,0.150930
6,n0062826215fe6aa,575,0.500315,0.500024,0.235694,0.604077,0.539700,0.235694,0.459824
7,n008361ac9e9bd47,575,0.498082,0.499196,0.027718,0.060980,0.144671,0.027718,0.077790
8,n009e95486e1d64c,575,0.501553,0.500474,0.372496,0.893598,0.760551,0.372496,0.675548
9,n00b093a02b84295,575,0.502781,0.500180,0.140916,0.991595,0.621245,0.140916,0.584585


In [9]:
# ============================================
# 5. 로컬 Numerai CORR / Sharpe
# ============================================

per_era_rows = []

for era, group in prediction_df.loc[
    prediction_df["target"].notna()
].groupby("era", sort=True):

    score = numerai_corr(
        group[["prediction"]],
        group["target"],
    )

    score_value = float(np.asarray(score).reshape(-1)[0])

    per_era_rows.append({
        "era": int(era),
        "corr": score_value,
        "rows": len(group),
    })

per_era_df = pd.DataFrame(per_era_rows)
per_era_df.to_csv(PER_ERA_CSV, index=False)

corr_mean = per_era_df["corr"].mean()
corr_std = per_era_df["corr"].std(ddof=0)
corr_sharpe = corr_mean / corr_std

cumulative_corr = per_era_df["corr"].cumsum()
max_drawdown = (
    cumulative_corr.expanding(min_periods=1).max() - cumulative_corr
).max()

summary_df = pd.DataFrame({
    "metric": [
        "mean_corr",
        "std_corr",
        "sharpe",
        "max_drawdown",
        "eras",
    ],
    "value": [
        corr_mean,
        corr_std,
        corr_sharpe,
        max_drawdown,
        len(per_era_df),
    ],
})

summary_df.to_csv(SUMMARY_CSV, index=False)

display(summary_df)

print("Per-era CSV:", PER_ERA_CSV)
print("Summary CSV:", SUMMARY_CSV)

,metric,value
0,mean_corr,0.013953
1,std_corr,0.015403
2,sharpe,0.905891
3,max_drawdown,0.239883
4,eras,649.000000


Per-era CSV: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/diagnostics_v52/elasticnet_et_rf_per_era_corr.csv
Summary CSV: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/diagnostics_v52/elasticnet_et_rf_local_summary.csv


In [ ]:
# ============================================
# 6. 최종 저장 결과 확인
# ============================================

final_check = pd.read_csv(DIAGNOSTICS_CSV)

print("파일 존재:", DIAGNOSTICS_CSV.exists())
print("행 수:", len(final_check))
print("컬럼:", final_check.columns.tolist())
print("중복 ID:", final_check["id"].duplicated().sum())
print("결측 prediction:", final_check["prediction"].isna().sum())

display(final_check.head())

파일 존재: True
행 수: 4050241
컬럼: ['id', 'prediction']
중복 ID: 0
결측 prediction: 0


,id,prediction
0,n000101811a8a843,0.534931
1,n001e1318d5072ac,0.903732
2,n002a9c5ab785cbb,0.506378
3,n002ccf6d0e8c5ad,0.685801
4,n0041544c345c91d,0.709347
